# Social Behavior Processing Pipeline

**Project:** Social Single/Group Housing Behavior Analysis  
**Author:** Mir Qi  
**Last Updated:** October 20, 2024

## Overview
This notebook processes and validates multi-camera behavioral recordings for social interaction experiments. The pipeline includes:
1. Scanning and logging folder structures
2. Reading and consolidating parquet logs
3. Data quality checks and filtering
4. Visualization of COM (center of mass) trajectories
5. DANNCE 3D pose validation

---
## 1. Setup & Scan Folders

Scan the data directory structure and create/update parquet log files for each session.

In [3]:
import sys
import os
sys.path.append(os.path.abspath('../..'))

from status_fields.status_fields_config_oct3v1_brws_250525 import STATUS_FIELDS_CONFIG

from utlis.scan_engine_utlis.scan_eng_big_utlis import log_folder_to_parquet_sep

base_folder = "/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused"
failed_paths_file = "/data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/bad_calib.txt"
force_rescan_rec_files = [
    # ('2023-10-01', '001'),
    # ('2023-10-02', '002'),
    # Add more as needed
]
rescan_threshold_days = 0.000000001 # 7 days, but guess if i mess up i can just change it to automatically rescan all, smile... #0.1

log_folder_to_parquet_sep(base_folder, failed_paths_file, STATUS_FIELDS_CONFIG,
                            force_rescan_rec_files=force_rescan_rec_files,
                            rescan_threshold_days=rescan_threshold_days)

Log for 0single5_group5_ saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5_/folder_log.parquet
Log for 0single5_group5 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5/folder_log.parquet
Log for 0single5_group5_3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_08_04/0single5_group5_3/folder_log.parquet
Log for extrinsics saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_03_17_00/extrinsics/folder_log.parquet
Log for 0single5_group3 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_17/0single5_group3/folder_log.parquet
Log for 0single4_group4 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_10_17/0single4_group4/folder_log.parquet
Log for 0single1_group1 saved at /data/big_rim/rsync_dcc_sum/minji_social_single_grouphoused/2025_09_26/0single1_group1/folder_log.parquet
Log for 0single2_group2 s

---
## 2. Load Data

Read all parquet logs - keep PyArrow table for filtering, create pandas for display.

In [4]:
sys.path.append(os.path.abspath('../..'))
from utlis.scan_engine_utlis.scan_engine_utlis import read_all_parquet_files

# Load as PyArrow table for efficient filtering
all_table = read_all_parquet_files(base_folder)

# Convert to pandas for display/visualization
all_df = all_table.to_pandas()

---
## 3. Data Overview

Display the loaded data and processing status summary.

In [5]:
print(all_df)

   mir_generate_param  ...                                        calib_files
0                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
1                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
2                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
3                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
4                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
5                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
6                   1  ...  [calib_after, calib_14_25_newintrinsics, calib...
7                   0  ...  [calib_after, calib_before_newintrinsics, cali...
8                   1  ...  [calib_after, calib_before_newintrinsics, cali...
9                   1  ...  [calib_after, calib_before_newintrinsics, cali...
10                  1  ...  [calib_before_newintrinsics, calib_before, cal...
11                  1  ...  [calib_before_newintrinsics, calib_b

In [6]:
all_df.columns.tolist()

['mir_generate_param',
 'sync',
 'mini_6cam_map',
 'dropf_handle',
 'com',
 'com_vis',
 'social',
 'miniscope',
 'test',
 'after_oxytocin',
 'before_oxytocin',
 'dannce',
 'dannce_vis',
 'mini_rec_sync',
 'mini_rec_sync_com',
 'rec_file',
 'scan_time',
 'rec_path',
 'date_folder',
 'calib_files']

In [7]:
# Check which columns exist in your DataFrame
print("Columns in all_df:", all_df.columns.tolist())

Columns in all_df: ['mir_generate_param', 'sync', 'mini_6cam_map', 'dropf_handle', 'com', 'com_vis', 'social', 'miniscope', 'test', 'after_oxytocin', 'before_oxytocin', 'dannce', 'dannce_vis', 'mini_rec_sync', 'mini_rec_sync_com', 'rec_file', 'scan_time', 'rec_path', 'date_folder', 'calib_files']


---
## 4. Filter Sessions

Apply filters to select sessions for processing. **Use PyArrow table for filtering!**

In [9]:
import pyarrow.compute as pc

# FILTER ON PYARROW TABLE (not pandas)
# NOTE: Values are stored as STRINGS in the parquet files
mask = (
    pc.equal(all_table['social'], '1'),
    pc.equal(all_table['com'], '1'),
    pc.equal(all_table['com_vis'], '0')
)

# Apply filter to get PyArrow table
filtered_table = all_table.filter(mask)

# NOW convert to pandas for display
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions: {len(filtered_df)}")
filtered_df

TypeError: Got unexpected argument type <class 'tuple'> for compute function

In [ ]:
# Alternative filter: social=1, com=0, com_vis=0
mask2 = (
    pc.equal(all_table['social'], '1') &
    pc.equal(all_table['com'], '0') &
    pc.equal(all_table['com_vis'], '0')
)
filtered_table = all_table.filter(mask2)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions (alternate): {len(filtered_df)}")
filtered_df

In [ ]:
# DANNCE filter: dannce=1, dannce_vis=0
mask3 = (
    pc.equal(all_table['dannce'], '1') &
    pc.equal(all_table['dannce_vis'], '0')
)
filtered_table = all_table.filter(mask3)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions (DANNCE): {len(filtered_df)}")
filtered_df

---
## 5. Advanced Filtering Examples

More complex filtering scenarios using PyArrow compute functions.

In [ ]:
# Filter: social=1, com=1
mask4 = (
    pc.equal(all_table['social'], '1') &
    pc.equal(all_table['com'], '1')
)
filtered_table = all_table.filter(mask4)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions: {len(filtered_df)}")
filtered_df

In [ ]:
# Filter: social=1, com=0
mask5 = (
    pc.equal(all_table['social'], '1') &
    pc.equal(all_table['com'], '0')
)
filtered_table = all_table.filter(mask5)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions: {len(filtered_df)}")
filtered_df

In [ ]:
# Filter: before_oxytocin=1
mask6 = pc.equal(all_table['before_oxytocin'], '1')
filtered_table = all_table.filter(mask6)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions: {len(filtered_df)}")
filtered_df

In [ ]:
# Filter: after_oxytocin=1
mask7 = pc.equal(all_table['after_oxytocin'], '1')
filtered_table = all_table.filter(mask7)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions: {len(filtered_df)}")
filtered_df

In [ ]:
# Filter by date folder pattern
mask8 = pc.match_substring(all_table['date_folder'], '2025_07')
filtered_table = all_table.filter(mask8)
filtered_df = filtered_table.to_pandas()

print(f"Filtered sessions (July 2025): {len(filtered_df)}")
filtered_df

---
## 6. Visualize Session Timeline

Show timeline of sessions with preview images.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from datetime import datetime

# 1. Get all full paths from filtered_df (pandas DataFrame)
full_paths = [
    os.path.join(base_folder, date_folder, rec_file)
    for date_folder, rec_file in zip(filtered_df['date_folder'], filtered_df['rec_file'])
]

# 2. Build (path, mtime) tuples
sessions = []
for p in full_paths:
    ft = os.path.join(p, "videos", "Camera1", "frametimes.mat")
    if os.path.isfile(ft):
        mtime = os.path.getmtime(ft)
        sessions.append((p, mtime))
    else:
        print(f"Warning: frametimes.mat not found for {p}")

# 3. Sort all sessions by timestamp
sessions.sort(key=lambda x: x[1])

# 4. Plot every session on a single figure
n = len(sessions)
if n == 0:
    print("No sessions to visualize")
else:
    fig, axes = plt.subplots(1, n, figsize=(n * 3, 3))
    if n == 1:
        axes = [axes]

    for ax, (p, mtime) in zip(axes, sessions):
        img_file = os.path.join(p, "COM/predict00/vis", "com_circle.png")
        if os.path.isfile(img_file):
            img = mpimg.imread(img_file)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "no image", ha="center", va="center")
        ts = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")
        ax.set_title(ts, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

---
## 7. Alternative Timeline Visualization

Using session images instead of COM visualizations.

In [ ]:
def visualize_sessions_timeline(table_or_df, base_folder, max_sessions=15):
    """
    Visualize sessions in chronological order with preview images.
    
    Args:
        table_or_df: PyArrow Table or Pandas DataFrame
        base_folder: Base path to data
        max_sessions: Maximum sessions to display
    """
    # Convert to pandas if needed for visualization
    if hasattr(table_or_df, 'to_pandas'):
        df = table_or_df.to_pandas()
    else:
        df = table_or_df
    
    if len(df) == 0:
        print("No sessions to visualize")
        return
    
    # Limit number of sessions
    df_display = df.head(max_sessions).copy()
    
    # Sort by scan_time
    df_display = df_display.sort_values('scan_time')
    
    # Create figure
    n_sessions = len(df_display)
    cols = 5
    rows = (n_sessions + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(15, 3*rows))
    axes = axes.flatten() if n_sessions > 1 else [axes]
    
    for idx, (_, row) in enumerate(df_display.iterrows()):
        ax = axes[idx]
        
        rec_name = row['rec_file']
        date_folder = row['date_folder']
        mtime = row['scan_time']
        
        # Try to find preview image
        session_path = os.path.join(base_folder, date_folder, rec_name)
        img_path = os.path.join(session_path, "imgs", "cam1_frame_000000.jpg")
        
        if os.path.isfile(img_path):
            img = mpimg.imread(img_path)
            ax.imshow(img)
        else:
            ax.text(0.5, 0.5, "No Image", ha="center", va="center", 
                   bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
        
        timestamp = datetime.fromtimestamp(mtime).strftime("%Y-%m-%d\n%H:%M")
        ax.set_title(f"{rec_name}\n{timestamp}", fontsize=8)
        ax.axis("off")
    
    # Hide unused subplots
    for idx in range(n_sessions, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.suptitle("Session Timeline (Chronological)", fontsize=14, y=1.02)
    plt.show()

# Visualize filtered sessions
visualize_sessions_timeline(filtered_df, base_folder, max_sessions=15)

---
## 8. Generate COM Visualizations

Process filtered sessions to generate center-of-mass trajectory visualizations for social experiments.

In [ ]:
# social com vis

# from utlis.vis_valid_utlis.com_trag_updated import plot_com_all
from utlis.vis_valid_utlis.scom_traga_utlis import plot_com_all_social

# Use filtered_table (PyArrow) for efficient iteration
for_com_vis = filtered_table

# Convert PyArrow table to list of records
records = [
    {
        'date_folder': date_folder.as_py(),  # Convert to string using as_py()
        'rec_file': rec_file.as_py()         # Convert to string using as_py()
    }
    for date_folder, rec_file in zip(for_com_vis['date_folder'], for_com_vis['rec_file'])
]

print(f"Processing {len(records)} sessions for COM visualization...\n")

# Iterate through the records and process each one sequentially
for i, record in enumerate(records, 1):
    base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
    print(f"[{i}/{len(records)}] {base_path}")
    
    try:
        plot_com_all_social(base_path, perform_generate_com_video=True)
        print("  ✓ Complete")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

print("\n=== COM visualization complete ===")

---
## 9. Validate DANNCE 3D Poses

Run validation on DANNCE 3D pose estimation results.

In [ ]:
# single dannce valid

from useful_files.sophie_check_dannce_mir_modif import dannce_valid

# Use filtered_table (PyArrow) for DANNCE processing
for_dannce_vis = filtered_table

# Convert PyArrow table to list of records
records = [
    {
        'date_folder': date_folder.as_py(),  # Convert to string using as_py()
        'rec_file': rec_file.as_py()         # Convert to string using as_py()
    }
    for date_folder, rec_file in zip(for_dannce_vis['date_folder'], for_dannce_vis['rec_file'])
]

print(f"Processing {len(records)} sessions for DANNCE validation...\n")

# Sequential processing (uncomment if needed)
# for i, record in enumerate(records, 1):
#     base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
#     print(f"[{i}/{len(records)}] {base_path}")
#     try:
#         dannce_valid(base_path)
#         print("  ✓ Complete")
#     except Exception as e:
#         print(f"  ✗ Error: {e}")
#         continue

# Parallel processing
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_record(record):
    base_path = f"{base_folder}/{record['date_folder']}/{record['rec_file']}"
    print(base_path)
    try:
        dannce_valid(base_path)
        return f"✓ {base_path}"
    except Exception as e:
        return f"✗ {base_path}: {e}"

with ProcessPoolExecutor() as executor:
    futures = [executor.submit(process_record, record) for record in records]
    for future in as_completed(futures):
        result = future.result()
        print(result)

print("\n=== DANNCE validation complete ===")

---
## Notes & Documentation

### Data Type Strategy
- **PyArrow Table** (`all_table`, `filtered_table`): Used for efficient filtering operations
- **Pandas DataFrame** (`all_df`, `filtered_df`): Used for display and visualization
- Use `.as_py()` when iterating over PyArrow tables to convert values to Python types
- Use `.to_pandas()` to convert PyArrow tables to pandas DataFrames for visualization
- **IMPORTANT**: Status field values are stored as STRINGS ('0', '1') not integers (0, 1)

### Key Files
- **STATUS_FIELDS_CONFIG**: Defines processing pipeline stages
- **bad_calib.txt**: List of sessions with calibration issues

### Processing Stages
1. `mir_generate_param` - Mirror parameter generation
2. `sync` - Camera synchronization
3. `mini_6cam_map` - 6-camera mapping
4. `dropf_handle` - Dropped frame handling
5. `com` - Center of mass calculation
6. `com_vis` - COM visualization
7. `social` - Social interaction flags
8. `dannce` - 3D pose estimation
9. `dannce_vis` - DANNCE visualization

### Filtering Examples
```python
# Filter using PyArrow compute
import pyarrow.compute as pc

# Equal condition - NOTE: Use STRINGS for status fields!
mask = pc.equal(all_table['column'], '1')  # NOT 1 (integer)

# Multiple conditions (AND)
mask = pc.equal(all_table['col1'], '1') & pc.equal(all_table['col2'], '0')

# String matching for date folders
mask = pc.match_substring(all_table['date_folder'], '2025_07')

# Apply filter
filtered_table = all_table.filter(mask)
```

### Troubleshooting
- If sessions are missing, check file permissions
- For failed visualizations, verify image paths
- Parallel processing may require adjusting worker count based on system resources
- Always filter on PyArrow tables, convert to pandas only for display
- **ArrowNotImplementedError**: Make sure you're comparing with strings ('0', '1') not integers